In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv('/home1/bingi' + '/mimic_iv_after_preprocessed_train.csv')
test = pd.read_csv('/home1/bingi' + '/mimic_iii_after_preprocessed_test.csv')

df.head()

,Unnamed: 0,key_0,first_careunit_Surgical Intensive Care Unit (SICU),last_careunit_Cardiac Vascular Intensive Care Unit (CVICU),last_careunit_Coronary Care Unit (CCU),last_careunit_Medical Intensive Care Unit (MICU),last_careunit_Medical/Surgical Intensive Care Unit (MICU/SICU),last_careunit_Neuro Intermediate,last_careunit_Neuro Stepdown,last_careunit_Neuro Surgical Intensive Care Unit (Neuro SICU),...,max_mean_BP,max_Spo2,min_Heart Rate,min_systolic_BP,min_diastolic_BP,min_mean_BP,min_Spo2,elix_score,SOFA_score,is_stroke
0,0,0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.030983,-0.008440,-0.724344,0.064523,0.638435,0.238671,0.440757,-0.731203,-0.826270,0
1,1,1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.021330,-0.024971,-0.525181,1.306274,0.121995,0.319263,0.288853,-0.731203,-0.826270,1
2,2,2,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.022709,-0.016705,-0.458794,-0.853292,-1.255179,-0.970205,0.288853,-0.484406,1.963532,0
3,3,3,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.001970,-0.008440,0.537019,0.077202,0.172049,0.120167,-0.014954,1.489976,-0.826270,0
4,4,4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.015125,-0.008440,1.665606,-0.907281,0.035921,0.238671,0.136949,1.613375,-0.826270,1


In [3]:
test.head()

,Unnamed: 0,key_0,first_careunit_Surgical Intensive Care Unit (SICU),last_careunit_Cardiac Vascular Intensive Care Unit (CVICU),last_careunit_Coronary Care Unit (CCU),last_careunit_Medical Intensive Care Unit (MICU),last_careunit_Medical/Surgical Intensive Care Unit (MICU/SICU),last_careunit_Neuro Intermediate,last_careunit_Neuro Stepdown,last_careunit_Neuro Surgical Intensive Care Unit (Neuro SICU),...,max_mean_BP,max_Spo2,min_Heart Rate,min_systolic_BP,min_diastolic_BP,min_mean_BP,min_Spo2,elix_score,SOFA_score,is_stroke
0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.769714,0.496166,0.144940,-0.155336,0.040062,-0.070488,0.080147,1.489976,-0.128819,0
1,1,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.618316,0.932401,-0.126856,-0.799303,-0.910886,-1.131389,1.200275,3.464359,1.266082,0
2,2,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.729892,-1.284725,0.043822,-0.085661,-0.247551,-0.165916,-0.223906,0.502785,1.963532,0
3,3,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.246193,0.932401,0.337856,-0.799303,-0.996959,-0.809022,0.288853,0.132589,-0.128819,0
4,4,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.788111,-4.461103,0.292294,-0.399338,0.001191,-0.185881,0.368478,-0.978001,-0.826270,0


In [4]:
df = df.drop(columns=['Unnamed: 0'], errors = 'ignore')
df = df.set_index('key_0')

df.columns

Index(['first_careunit_Surgical Intensive Care Unit (SICU)',
       'last_careunit_Cardiac Vascular Intensive Care Unit (CVICU)',
       'last_careunit_Coronary Care Unit (CCU)',
       'last_careunit_Medical Intensive Care Unit (MICU)',
       'last_careunit_Medical/Surgical Intensive Care Unit (MICU/SICU)',
       'last_careunit_Neuro Intermediate', 'last_careunit_Neuro Stepdown',
       'last_careunit_Neuro Surgical Intensive Care Unit (Neuro SICU)',
       'last_careunit_Surgical Intensive Care Unit (SICU)',
       'last_careunit_Trauma SICU (TSICU)',
       ...
       'max_mean_BP', 'max_Spo2', 'min_Heart Rate', 'min_systolic_BP',
       'min_diastolic_BP', 'min_mean_BP', 'min_Spo2', 'elix_score',
       'SOFA_score', 'is_stroke'],
      dtype='object', length=107)

In [5]:
test = test.drop(columns=['Unnamed: 0'], errors = 'ignore')
test = test.set_index('key_0')

l = pd.DataFrame(
    test.columns)
l.head(101)

,0
0,first_careunit_Surgical Intensive Care Unit (S...
1,last_careunit_Cardiac Vascular Intensive Care ...
2,last_careunit_Coronary Care Unit (CCU)
3,last_careunit_Medical Intensive Care Unit (MICU)
4,last_careunit_Medical/Surgical Intensive Care ...
...,...
96,max_diastolic_BP
97,max_mean_BP
98,max_Spo2
99,min_Heart Rate


In [6]:
y_train = df['is_stroke'].astype(int)
X_train = df.drop(columns=['is_stroke'])

In [7]:
y_test = test['is_stroke'].astype(int)
X_test = test.drop(columns=['is_stroke'])



In [8]:
from sklearn.feature_selection import RFECV
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
"""

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)
"""
c = StratifiedKFold(5)



from xgboost import XGBClassifier
from sklearn.feature_selection import RFECV
from sklearn.model_selection import StratifiedKFold

scale_pos_weight = (len(y_train) - sum(y_train)) / sum(y_train)

xgb1 = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    eval_metric='auc',
    random_state=42
)

rfecv = RFECV(
    estimator=xgb1,
    cv=c,
    scoring='accuracy',
    n_jobs=-1
)

#rfecv.fit(X_train, y_train)

In [9]:
rfecv.fit(X_train, y_train)

selected_features = X_train.columns[rfecv.support_]

print("Optimal number of features:", rfecv.n_features_)
print("Selected Features:", list(selected_features))

#X_train_rfecv = X_train[selected_features]
#X_test_rfecv  = X_test[selected_features]

y_pred = rfecv.predict(X_test)
print("Accuracy on test set:", accuracy_score(y_test, y_pred))

Optimal number of features: 80
Selected Features: ['last_careunit_Medical Intensive Care Unit (MICU)', 'last_careunit_Neuro Intermediate', 'last_careunit_Neuro Stepdown', 'last_careunit_Neuro Surgical Intensive Care Unit (Neuro SICU)', 'last_careunit_Surgical Intensive Care Unit (SICU)', 'ethnicity_ASIAN', 'ethnicity_BLACK/AFRICAN AMERICAN', 'ethnicity_UNKNOWN', 'ethnicity_WHITE', 'gender_F', 'los', 'age', 'htn', 'renal', 'liver', 'pud', 'sepsis', 'chf', 'cancer', 'Alcohol', 'Tobacco', 'mean_Anion Gap', 'mean_Ca', 'mean_chloride', 'mean_Creatinine', 'mean_Glucose', 'mean_Mg', 'mean_Phosphate', 'mean_Potassium', 'mean_Na', 'mean_Urea Nitrogen', 'mean_INR(PT)', 'mean_PT', 'mean_PTT', 'mean_WBC', 'max_Anion Gap', 'max_Ca', 'max_chloride', 'max_Creatinine', 'max_Glucose', 'max_Mg', 'max_Phosphate', 'max_Potassium', 'max_Na', 'max_Urea Nitrogen', 'max_INR(PT)', 'max_Platelet Count', 'max_PT', 'max_PTT', 'max_WBC', 'min_Anion Gap', 'min_Ca', 'min_chloride', 'min_Creatinine', 'min_Glucose', '

In [10]:
feature_ranking = pd.DataFrame({
    'feature': X_test.columns,
    'ranking': rfecv.ranking_
})

feature_ranking_sorted = feature_ranking.sort_values(by='ranking')
print(feature_ranking_sorted[feature_ranking_sorted['ranking']==1])

                                              feature  ranking
3    last_careunit_Medical Intensive Care Unit (MICU)        1
7   last_careunit_Neuro Surgical Intensive Care Un...        1
6                        last_careunit_Neuro Stepdown        1
5                    last_careunit_Neuro Intermediate        1
15                   ethnicity_BLACK/AFRICAN AMERICAN        1
..                                                ...      ...
91                                  mean_diastolic_BP        1
92                                       mean_mean_BP        1
93                                          mean_Spo2        1
94                                     max_Heart Rate        1
95                                    max_systolic_BP        1

[80 rows x 2 columns]


In [11]:
#_____________________++++++++++++++++++++++++++

selected_features = ['last_careunit_Medical Intensive Care Unit (MICU)', 'last_careunit_Neuro Intermediate', 
                    'last_careunit_Neuro Stepdown', 'last_careunit_Neuro Surgical Intensive Care Unit (Neuro SICU)', 
                    'last_careunit_Surgical Intensive Care Unit (SICU)', 'ethnicity_ASIAN', 'ethnicity_BLACK/AFRICAN AMERICAN', 
                    'ethnicity_UNKNOWN', 'ethnicity_WHITE', 'gender_F', 'los', 'age', 'htn', 'renal', 'liver', 'pud', 
                    'sepsis', 'chf', 'cancer', 'Alcohol', 'Tobacco', 'mean_Anion Gap', 'mean_Ca', 'mean_chloride', 
                    'mean_Creatinine', 'mean_Glucose', 'mean_Mg', 'mean_Phosphate', 'mean_Potassium', 'mean_Na', 
                    'mean_Urea Nitrogen', 'mean_INR(PT)', 'mean_PT', 'mean_PTT', 'mean_WBC', 'max_Anion Gap', 'max_Ca', 
                    'max_chloride', 'max_Creatinine', 'max_Glucose', 'max_Mg', 'max_Phosphate', 'max_Potassium', 'max_Na', 
                    'max_Urea Nitrogen', 'max_INR(PT)', 'max_Platelet Count', 'max_PT', 'max_PTT', 'max_WBC', 'min_Anion Gap', 
                    'min_Ca', 'min_chloride', 'min_Creatinine', 'min_Glucose', 'min_Mg', 'min_Phosphate', 'min_Potassium',
                    'min_Na', 'min_Urea Nitrogen', 'min_INR(PT)', 'min_Platelet Count', 'min_PT', 'min_PTT', 'min_WBC', 
                    'mean_Heart Rate', 'mean_systolic_BP', 'mean_diastolic_BP', 'mean_mean_BP', 'mean_Spo2', 'max_Heart Rate', 
                    'max_systolic_BP', 'max_diastolic_BP', 'max_mean_BP', 'min_Heart Rate', 'min_systolic_BP', 'min_mean_BP', 
                    'min_Spo2', 'elix_score', 'SOFA_score']


X_train_rfe = X_train[selected_features]
X_test_rfe = X_test[selected_features]

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=500,
    random_state=42,
    class_weight="balanced",  
    n_jobs=-1
)

rf.fit(X_train_rfe, y_train)


In [ ]:
from sklearn.inspection import permutation_importance

result = permutation_importance(
    rf,
    X_test_rfe,
    y_test,
    scoring='roc_auc',
    n_repeats=10,
    random_state=42
)

perm_importance = pd.Series(
    result.importances_mean,
    index=X_train_rfe.columns
).sort_values(ascending=False)

print(perm_importance)


In [ ]:
final_features = perm_importance.head(25).index

In [ ]:
#xgb2.fit(X_train_rfe, y_train)
final_features


In [ ]:
import shap

explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test_rfe)



In [ ]:
print(rf.n_features_in_)
print(X_test_rfe.shape[1])  # or X_test_rfecv.shape[1]
print(len(shap_values))
print(len(X_test_rfe))
print(type(shap_values))
print(np.array(shap_values).shape)


In [ ]:
# Global importance
shap.summary_plot(shap_values[:, :, 1], X_test_rfe, max_display=30)


In [ ]:

final_features = ['elix_score', 'min_systolic_BP', 'los', 'cancer',
                'age', 'max_systolic_BP', 'min_Heart Rate', 'gender_F',
                'max_Ca', 'mean_Urea Nitrogen', 'max_Potassium', 'mean_mean_BP',
                'SOFA_score', 'min_Platelet Count', 'mean_PTT', 'mean_WBC',
                'mean_Phosphate', 'liver', 'chf', 'htn',
                   'mean_Anion Gap']

In [ ]:
"""final_features = ['los', 'age', 'mean_Ca', 'mean_Phosphate', 
                  'mean_Potassium', 'mean_Urea Nitrogen', 'mean_PT', 
                  'mean_PTT', 'max_Ca', 'max_Phosphate', 'max_Potassium', 
                  'max_Urea Nitrogen', 'max_PT', 'min_Ca', 'max_INR(PT)', 'cancer',
                  'min_Platelet Count', 'min_PT', 'min_WBC', 'max_Urea Nitrogen',
                  'mean_Heart Rate', 'mean_systolic_BP', 'mean_diastolic_BP', 'mean_mean_BP', 
                  'mean_Spo2', 'max_Heart Rate', 'max_systolic_BP', 
                  'max_mean_BP', 'min_Heart Rate', 'min_systolic_BP',
                  'min_Spo2', 'elix_score', 'SOFA_score']


'last_careunit_Surgical Intensive Care Unit (SICU)', 'elix_score', 'los', 
final_features = ['los', 'age','cancer','mean_Ca', 'mean_Phosphate', 
                 'mean_Na', 'mean_Urea Nitrogen', 
                  'max_Ca', 'max_Phosphate', 'max_Na', 
                  'max_Urea Nitrogen', 'min_Ca',  
                  'mean_Potassium', 'min_Platelet Count', 'mean_systolic_BP', 
                  'max_Heart Rate', 'max_systolic_BP', 'max_diastolic_BP', 'max_mean_BP', 
                  'min_systolic_BP', 'elix_score', 'SOFA_score']
"""

"""final_features = ['los', 'age', 'sepsis', 'mean_Ca', 'max_chloride', 'max_Glucose',
       'max_Mg', 'max_Phosphate', 'max_Urea Nitrogen', 'max_WBC',
       'min_Anion Gap', 'min_chloride', 'min_Mg', 'min_Potassium', 'min_WBC',
       'mean_Heart Rate', 'mean_systolic_BP', 'mean_Spo2', 'min_Heart Rate',
       'elix_score']

final_features = ['last_careunit_Cardiac Vascular Intensive Care Unit (CVICU)',
       'last_careunit_Coronary Care Unit (CCU)',
       'last_careunit_Medical Intensive Care Unit (MICU)',
       'last_careunit_Medical/Surgical Intensive Care Unit (MICU/SICU)',
       'insurance_Other', 'ethnicity_ASIAN', 'ethnicity_OTHER',
       'ethnicity_UNABLE TO OBTAIN', 'ethnicity_UNKNOWN', 'los', 'age', 'pud',
       'sepsis', 'cancer', 'Alcohol', 'Opioid', 'mean_Ca',
       'mean_Urea Nitrogen', 'mean_INR(PT)', 'mean_WBC', 'max_chloride',
       'max_Glucose', 'max_Mg', 'max_Phosphate', 'max_Potassium',
       'max_Urea Nitrogen', 'max_WBC', 'min_Anion Gap', 'min_Phosphate',
       'min_Potassium', 'min_Urea Nitrogen', 'min_Platelet Count', 'min_PTT',
       'min_WBC', 'mean_systolic_BP', 'mean_Spo2', 'max_Spo2',
       'min_Heart Rate', 'elix_score', 'SOFA_score']
"""
X_train = X_train[final_features]
X_test = X_test[final_features]

len(final_features)

In [ ]:
train = X_train.merge(y_train, how='left', on=X_train.index)
test = X_test.merge(y_test, how='left', on=X_test.index)

In [ ]:
#df = pd.concat([train, test], axis=0, ignore_index=True)

In [ ]:
num_cols = train.select_dtypes(include=['float64','int64']).columns
cat_cols = train.select_dtypes(include=['object','category']).columns

In [ ]:
import seaborn as sns

def plot_pearson_correlation(df, num_cols):
    corr = df[num_cols].corr()

    plt.figure(figsize=(84,80))
    sns.heatmap(corr, cmap='coolwarm', square=True)
    plt.title("Pearson Correlation Matrix")
    plt.show()

    high_corr_pairs = (
    corr.abs()
    .where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    .stack()
    .sort_values(ascending=False)
    )

    tab = high_corr_pairs[high_corr_pairs > 0.8]
    tab.to_csv('correlation.csv')
    return corr


In [ ]:
plot_pearson_correlation(train, num_cols)

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd
import numpy as np

def compute_vif(df, num_features):
    X = df[num_features].dropna()

    vif_data = pd.DataFrame()
    vif_data["Feature"] = X.columns
    vif_data["VIF"] = [
        variance_inflation_factor(X.values, i)
        for i in range(X.shape[1])
    ]

    return vif_data.sort_values("VIF", ascending=False)


vif_table = compute_vif(train, num_cols)
print(vif_table)

In [ ]:
def correlation_with_target(df, num_features, target='is_stroke'):
    clean_cols = list(set(num_features + [target]).intersection(df.columns))

    corr_with_series = df[clean_cols] \
                      .corr()[target] \
                      .drop(target) \
                      .sort_values(key=abs, ascending=False)


    return corr_series

target_corr = correlation_with_target(train, num_cols)
print(target_corr)

In [ ]:
train.to_csv('mimic_iv_train_model_ready.csv')
test.to_csv('mimic_iii_test_model_ready.csv')
